<a href="https://colab.research.google.com/github/ihor-stanishevskyi/bio-ai-journey/blob/main/00_mammal_protein_solubility_first_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
print("hello")

hello


In [8]:
!pip install -q git+https://github.com/BiomedSciAI/biomed-multi-alignment.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [9]:
from fuse.data.tokenizers.modular_tokenizer.op import ModularTokenizerOp
from mammal.examples.protein_solubility.task import ProteinSolubilityTask
from mammal.keys import CLS_PRED, SCORES
from mammal.model import Mammal
CKPT = "ibm/biomed.omics.bl.sm.ma-ted-458m.protein_solubility"
model = Mammal.from_pretrained(CKPT)
model.eval()
tokenizer_op = ModularTokenizerOp.from_pretrained(CKPT)
print("model loaded on", model.device)

Path doesn't exist. Will try to download from hf hub. pretrained_model_name_or_path='ibm/biomed.omics.bl.sm.ma-ted-458m.protein_solubility'


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Attempting to load model from dir: pretrained_model_name_or_path='/root/.cache/huggingface/hub/models--ibm--biomed.omics.bl.sm.ma-ted-458m.protein_solubility/snapshots/ec682e1e1e4dd769440732127294c00811e04775'


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model loaded on cpu


In [10]:
from mammal.examples.protein_solubility.pl_data_module import load_datasets
ds = load_datasets("./sol_data")
test = ds["test"]
print("device:", model.device, "| test size:", len(test))

train set size is 62478
val set size is 6942
test set size is 1999
device: cpu | test size: 1999


In [11]:
protein_seq = "NLMKRCTRGFRKLGKCTTLEEEKCKTLYPRGQCTCSDSKMNTHSCDCKSC"
sample_dict = {"protein_seq": protein_seq}
sample_dict = ProteinSolubilityTask.data_preprocessing(sample_dict=sample_dict, protein_sequence_key="protein_seq", tokenizer_op=tokenizer_op, device=model.device)
batch_dict = model.generate([sample_dict], output_scores=True, return_dict_in_generate=True, max_new_tokens=5)
ans = ProteinSolubilityTask.process_model_output(tokenizer_op=tokenizer_op, decoder_output=batch_dict[CLS_PRED][0], decoder_output_scores=batch_dict[SCORES][0])
print(ans)

{'pred': 1, 'not_normalized_scores': tensor(0.9977), 'normalized_scores': tensor(0.9977)}


In [12]:
def solubility(seq):
  d = {"protein_seq": seq}
  d = ProteinSolubilityTask.data_preprocessing(sample_dict=d, protein_sequence_key="protein_seq", tokenizer_op=tokenizer_op, device=model.device)
  out = model.generate([d], output_scores=True, return_dict_in_generate=True, max_new_tokens=5)
  return ProteinSolubilityTask.process_model_output(tokenizer_op=tokenizer_op, decoder_output=out[CLS_PRED][0], decoder_output_scores=out[SCORES][0])

In [13]:
insulin_b = "FVNQHLCGSHLVEALYLVCGERGFFYTPKT"
lysozyme = "KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL"
collagen_like = "GPPGPPGPPGPPGPPGPPGPPGPPGPPGPPGPPGPPGPPGPP"
print("insulin B  ", solubility(insulin_b))
print("lysozyme   ", solubility(lysozyme))
print("collagen   ", solubility(collagen_like))

insulin B   {'pred': 1, 'not_normalized_scores': tensor(0.8985), 'normalized_scores': tensor(0.8985)}
lysozyme    {'pred': 1, 'not_normalized_scores': tensor(0.9673), 'normalized_scores': tensor(0.9673)}
collagen    {'pred': 1, 'not_normalized_scores': tensor(0.9980), 'normalized_scores': tensor(0.9980)}


In [14]:
model = model.to("cuda")
from mammal.examples.protein_solubility.pl_data_module import load_datasets
ds = load_datasets("./sol_data")
test = ds["test"]
print("device:", model.device, "| test size:", len(test))

train set size is 62478
val set size is 6942
test set size is 1999
device: cuda:0 | test size: 1999


In [19]:
import random, numpy as np
_pre = lambda seq: ProteinSolubilityTask.data_preprocessing(sample_dict={"protein_seq": seq[:1200], "data.sample_id": 0}, protein_sequence_key="protein_seq", tokenizer_op=tokenizer_op, device=model.device)
_gen = lambda d: model.generate([d], output_scores=True, return_dict_in_generate=True, max_new_tokens=5)
_post = lambda o: ProteinSolubilityTask.process_model_output(tokenizer_op=tokenizer_op, decoder_output=o[CLS_PRED][0], decoder_output_scores=o[SCORES][0])
predict = lambda seq: (lambda r: (int(r["pred"]), float(r["normalized_scores"])))(_post(_gen(_pre(seq))))
print(predict("KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL"))

(1, 0.9672658443450928)


In [20]:
import pandas as pd, glob
p_src = glob.glob("./sol_data/**/data/test_src", recursive=True)[0]
p_tgt = p_src.replace("test_src", "test_tgt")
seqs = pd.read_csv(p_src, names=["seq"])["seq"].astype(str).tolist()
labs = pd.read_csv(p_tgt, names=["y"])["y"].astype(int).tolist()
print("loaded:", len(seqs), len(labs), "| label balance:", round(sum(labs)/len(labs), 3))
random.seed(0)
N = 100
idx = random.sample(range(len(seqs)), N)
res = [(labs[i], *predict(seqs[i])) for i in idx]
y = np.array([r[0] for r in res]); p = np.array([r[1] for r in res]); s = np.array([r[2] for r in res])
from sklearn.metrics import roc_auc_score
print("n:", N)
print("accuracy:", round(float((y == p).mean()), 3))
print("label positive rate:", round(float(y.mean()), 3), "| predicted positive rate:", round(float(p.mean()), 3))
print("TP", int(((y==1)&(p==1)).sum()), "TN", int(((y==0)&(p==0)).sum()), "FP", int(((y==0)&(p==1)).sum()), "FN", int(((y==1)&(p==0)).sum()))
print("AUROC:", round(float(roc_auc_score(y, s)), 3))
print("mean confidence:", round(float(np.maximum(s, 1-s).mean()), 3))

loaded: 1999 1999 | label balance: 0.5
n: 100
accuracy: 0.77
label positive rate: 0.42 | predicted positive rate: 0.45
TP 32 TN 45 FP 13 FN 10
AUROC: 0.86
mean confidence: 0.788


In [21]:
probes = {"poly-GPP synthetic": "GPP"*14, "poly-A": "A"*50, "poly-Q": "Q"*40, "random 150aa": "".join(random.choice("ACDEFGHIKLMNPQRSTVWY") for _ in range(150)), "amyloid-beta 42": "DAEFRHDSGYEVHHQKLVFFAEDVGSNKGAIIGLMVGGVVIA", "insulin B chain": insulin_b, "lysozyme": lysozyme}
for name, seq in probes.items(): pr, cf = predict(seq); print(f"{name:22s} pred={pr}  P(soluble)={cf:.3f}  len={len(seq)}")

poly-GPP synthetic     pred=1  P(soluble)=0.998  len=42
poly-A                 pred=1  P(soluble)=0.668  len=50
poly-Q                 pred=1  P(soluble)=0.976  len=40
random 150aa           pred=1  P(soluble)=0.696  len=150
amyloid-beta 42        pred=1  P(soluble)=0.887  len=42
insulin B chain        pred=1  P(soluble)=0.899  len=30
lysozyme               pred=1  P(soluble)=0.967  len=129
